# Lab 6 — End-to-End 3DGS Production Pipeline
### YouTube Video → COLMAP / GLOMAP → 3DGS Training → `.splat`

**Course**: 3D Gaussian Splatting Compression  
**Instructor**: Nicholas McCarty (Upskilled Consulting)

This notebook runs the **streamlined production pipeline** validated by Nicholas McCarty on Bamburgh Castle footage.
No depth priors are required — GLOMAP’s global SfM initialisation provides sufficient scene structure for 3DGS
training to converge to PSNR ≥ 29 dB in under 15 minutes on a Colab T4.

| # | Learning Objective | Notebook Section |
|---|---|---|
| LO1 | SfM as joint pose + point optimisation; GLOMAP global pipeline vs. incremental COLMAP | §3–4 SfM Pipeline |
| LO2 | Essential matrix, RANSAC, Ceres convergence diagnostics | §5 SfM Inspection |
| LO3 | Why 3DGS skips image undistortion; convert.py as historical artifact | §5 Commented-Out Code |
| LO4 | 3DGS training convergence; PSNR at 7k and 30k iterations | §6 Training |
| LO5 | 32-byte `.splat` record: SH₀, sort key, quaternion encoding | §8–9 Format & Sort Key |
| LO6 | Field-aware PLY dispatcher; compression ratio analysis | §10–11 Compression |

---

### Reference Run (Bamburgh Castle, Colab T4 — Streamlined)

| Stage | Duration | Key Output |
|---|---|---|
| Video download + frame extraction | ~2 min | 1,031 frames @ 640×272, 7 fps |
| COLMAP feature extraction | ~2 min | SIFT features from 1,031 images |
| COLMAP exhaustive matching | ~10 min | ~96k verified image pairs |
| GLOMAP global reconstruction | ~17 min | 512/1,031 cameras, 13,427 3D points |
| 3DGS training (30k iterations) | ~14 min | PSNR 29.63 dB |
| PLY → .splat conversion | ~10 s | 32 bytes/Gaussian |
| **Total** | **~45 min** | |

> **Extended workflow** (§Extended): Adding Depth-Anything-V2 depth priors and Sparse Adam optimisation costs
> ~50 extra minutes and yields PSNR 29.02 dB on the same footage — *lower* by this metric, but with different
> Gaussian density characteristics that can matter for thin structures and low-texture regions.

## §0 — Install Dependencies

We use **pre-built COLMAP and GLOMAP binaries** from `nickmccarty/3dgs-workshop` rather than compiling from
source or using the apt-packaged COLMAP. The apt COLMAP triggers a CUDA/Qt display SIGABRT in headless Colab;
the pre-built binaries are compiled against Colab’s exact CUDA stack and work reliably via `!` shell magic.

> **Why `!` and not `subprocess.run`?**  
> IPython’s `!` magic inherits the full Colab shell environment including CUDA library paths. `subprocess.run`
> with a list argument spawns a minimal subprocess that may not see all `LD_LIBRARY_PATH` entries — causing
> silent COLMAP failures where the database is written but remains empty.

In [ ]:
# ── Video ingestion
!pip install yt-dlp
!apt-get install -y ffmpeg

# ── Pre-built COLMAP + GLOMAP binaries
!wget https://github.com/nickmccarty/3dgs-workshop/raw/refs/heads/master/colmap.zip
!wget https://github.com/nickmccarty/3dgs-workshop/raw/refs/heads/master/glomap.zip
!unzip colmap.zip
!unzip glomap.zip

# ── COLMAP/GLOMAP runtime dependencies
!sudo apt-get install -y \
    libboost-program-options-dev libboost-filesystem-dev \
    libboost-graph-dev libboost-system-dev \
    libeigen3-dev libflann-dev libfreeimage-dev libmetis-dev \
    libgoogle-glog-dev libgtest-dev libsqlite3-dev libglew-dev \
    qtbase5-dev libqt5opengl5-dev libcgal-dev libceres-dev

!sudo apt-get install -y nvidia-cuda-toolkit nvidia-cuda-toolkit-gcc

# ── 3DGS
!pip install -q plyfile
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive
%cd gaussian-splatting
!pip install submodules/simple-knn
!pip install submodules/diff-gaussian-rasterization
%cd ..
%mkdir data

In [ ]:
import os
os.environ['PATH'] += ':/content/colmap/build/src/colmap/exe'
os.environ['PATH'] += ':/content/glomap-1.0.0/build/glomap'

In [ ]:
!colmap -h

## §1 — Pipeline Timer

Every stage is wrapped in a `stage()` context manager that records wall-clock elapsed time.
At the end of the pipeline, `timing_summary()` produces a DataFrame you can compare against the
reference timings in the header table above (**Lab Objective 1**).

In [ ]:
import time
import contextlib
import pandas as pd

_STAGES = []

@contextlib.contextmanager
def stage(name, expected_output=''):
    """Context manager that times a pipeline stage and logs it."""
    print(f"\n{'='*60}\n▶  {name}\n{'='*60}")
    t0 = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - t0
        _STAGES.append({'Stage': name, 'Elapsed (s)': round(elapsed, 1), 'Key Output': expected_output})
        print(f'   ✓ completed in {elapsed:.1f} s')

def timing_summary():
    df = pd.DataFrame(_STAGES)
    df['Elapsed (min)'] = (df['Elapsed (s)'] / 60).round(1)
    total = df['Elapsed (s)'].sum()
    print(df[['Stage', 'Elapsed (min)', 'Key Output']].to_string(index=False))
    print(f'\nTotal pipeline time: {total/60:.1f} min')
    return df

print('Timer ready.')

## §2 — Video Download & Frame Extraction

We download the Bamburgh Castle fly-through from YouTube using `yt-dlp`, then extract frames at **7 fps**
with `ffmpeg`. At 7 fps the 2:27 video yields 1,031 frames — enough temporal overlap for robust feature
matching without the computational cost of a higher frame rate.

All pipeline artifacts live under `data/` so that `train.py -s data` can locate them by convention:
```
data/
├── images/          ← input frames
├── database.db      ← COLMAP feature database
└── sparse/0/        ← GLOMAP reconstruction output
```

In [ ]:
from yt_dlp import YoutubeDL

URL = 'https://youtu.be/nefKdNDJR9M?si=4PDguQYUWTdTtZtG'

with stage('Video download', 'downloaded_video.mp4 (~6 MB)'):
    ydl_opts = {'format': 'best', 'outtmpl': './downloaded_video.%(ext)s'}
    with YoutubeDL(ydl_opts) as ydl:
        ydl.download([URL])

In [ ]:
from IPython.display import Video

# Display the local video
Video("downloaded_video.mp4", width=640, height=272, embed=True)

In [ ]:
with stage('Frame extraction — 7 fps', 'data/images/ — ~1,031 frames @ 640×272'):
    %cd data
    !mkdir images
    !ffmpeg -i ../downloaded_video.mp4 -qscale:v 1 -qmin 1 -vf "fps=7" images/image%04d.jpg

## §3 — COLMAP: Feature Extraction & Matching

COLMAP’s feature extractor detects SIFT keypoints and computes descriptors for every image.
The exhaustive matcher then computes descriptor distances for every image pair and stores verified matches
in `database.db`.

**Key flags explained:**

| Flag | Value | Rationale |
|---|---|---|
| `--ImageReader.camera_model` | `PINHOLE` | Single fixed lens; no radial distortion model needed |
| `--ImageReader.single_camera` | `1` | All frames share one camera instance → shared intrinsics during BA |
| `--SiftExtraction.use_gpu` | `1` | CUDA SIFT: ~10× faster than CPU on T4 (~2 min vs ~20 min) |
| `--SiftMatching.use_gpu` | `1` | CUDA matching: exhaustive 1,031×1,031 pairs in ~10 min vs ~60 min CPU |

Setting `single_camera 1` is important for video sources: it constrains focal length and principal point
to be identical across all frames, which is physically correct and prevents the optimizer from fitting
spurious per-image intrinsic variations.

In [ ]:
with stage('COLMAP feature extraction', 'database.db with SIFT keypoints from 1,031 images'):
    !colmap feature_extractor \
       --database_path database.db \
       --image_path images \
       --ImageReader.camera_model PINHOLE \
       --ImageReader.single_camera 1 \
       --SiftExtraction.use_gpu 1

In [ ]:
with stage('COLMAP exhaustive matching', '~96k verified image pairs in database.db'):
    !colmap exhaustive_matcher \
       --database_path database.db \
       --SiftMatching.use_gpu 1

### 📌 Discussion: Why Is `colmap mapper` Commented Out? (LO1)

```python
# !mkdir sparse
# !colmap mapper \
#     --database_path database.db \
#     --image_path images \
#     --output_path sparse \
#     --Mapper.ba_global_function_tolerance=0.000001
```

The original Gaussian Splatting paper pipeline used COLMAP’s **incremental mapper** (`colmap mapper`).
Incremental SfM works by:
1. Choosing a two-image seed pair with many verified matches
2. Triangulating an initial 3D point cloud
3. Greedily registering each new camera via PnP + RANSAC
4. Running local bundle adjustment after each registration

The problem: errors from early registrations accumulate — this is called **drift**. On a 1,031-frame fly-through,
incremental SfM can drift badly at the loop closure (the point where the camera revisits an earlier viewpoint).

**GLOMAP** (`glomap mapper`) replaces this with **global SfM**:
1. Build a view graph of pairwise relative rotations
2. Solve rotation averaging globally across all pairs simultaneously (minimising a single cost over all R)
3. Solve global camera positions in one pass
4. Run bundle adjustment to refine everything jointly

Global SfM has no drift because it never makes greedy sequential decisions. On the Bamburgh Castle dataset,
GLOMAP registers 512/1,031 cameras in ~17 minutes. The incremental mapper would take 60–90 minutes on CPU
and would accumulate visible drift in the castle towers.

**Module 6 LO1 connection**: This is the core architectural choice of the production pipeline — treating
SfM as a *joint* optimisation problem (rotation averaging + global positioning + bundle adjustment) rather
than a greedy sequential one.

## §4 — GLOMAP: Global Reconstruction

GLOMAP runs five stages (view graph calibration → relative pose estimation → rotation averaging →
global positioning → bundle adjustment) and writes a COLMAP-compatible binary reconstruction to `sparse/0/`.

### What GLOMAP Is Optimising

Every SfM pipeline ultimately solves the same problem: minimize the total **reprojection error** across all
cameras and 3D points jointly. This is the **bundle adjustment** (BA) objective:

$$\mathcal{E}_{\mathrm{BA}} = \sum_{i,j} \rho\!\left( \left\| \pi(K_i, R_i, \mathbf{t}_i, \mathbf{X}_j) - \mathbf{p}_{ij} \right\|^2 \right)$$

where $\pi$ is the projection function, $\mathbf{p}_{ij}$ is the observed 2D keypoint, and $\rho$ is a robust
loss (Huber or Cauchy) that down-weights outlier correspondences.

The Jacobian of $\mathcal{E}_{\mathrm{BA}}$ is large but **sparse**: each 3D point $\mathbf{X}_j$ only
appears in the cameras that observe it. The normal equations $J^\top J\,\delta = -J^\top r$ are solved
efficiently using the **Schur complement trick**:

$$\left( B - C A^{-1} C^\top \right) \delta_c = -(\mathbf{g}_c - C A^{-1} \mathbf{g}_p)$$

where $A$ is the block-diagonal point sub-matrix (trivially invertible block-by-block), $B$ is the camera
sub-matrix, and $C$ couples cameras to points. This reduces the solve from $O(n^3)$ in the full system to
$O(m^3)$ in the camera system — practical for thousands of cameras with millions of points.

The key difference from COLMAP’s incremental mapper is *when* BA runs: COLMAP triggers a new BA solve
after each camera registration (serial, drift accumulates); GLOMAP runs BA once globally after rotation
averaging and global positioning have already provided a good initialisation.

### Reference Convergence (Bamburgh Castle, T4)

- Relative pose estimation: ~8 min (91,804 pairs; 15,201 rejected for < 30 inliers)
- Global positioning Ceres: Initial cost $1.764 \times 10^6$ → Final cost $1.326 \times 10^1$ (CONVERGENCE, 59 iterations)
- Bundle adjustment: 3 rounds, final cost $5.987 \times 10^4$
- Registered: **512 / 1,031 cameras**, **13,427 3D points**

The 50% registration rate is typical for a forward-moving fly-through: frames at the start and end of the
sequence have little overlap with the middle, and GLOMAP correctly excludes them rather than forcing a bad
registration.


In [ ]:
with stage('GLOMAP global reconstruction', 'sparse/0/ — 512 cameras, ~13k 3D points'):
    !glomap mapper \
        --database_path database.db \
        --image_path images \
        --output_path sparse

In [ ]:
%cd ..

## §5 — SfM Output Inspection (**Lab Objective 2**)

GLOMAP writes its output in COLMAP binary format: `cameras.bin`, `images.bin`, `points3D.bin`.
We parse the binary files directly to:
1. Count registered cameras and compute the registration rate
2. visualize camera positions in 3D (the reconstructed trajectory)
3. Count 3D points and inspect bounding box

This satisfies **Lab Objective 2**: inspect SfM output, count registered cameras, visualize positions.

In [ ]:
import struct
import glob
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def read_images_binary(path):
    """Parse COLMAP images.bin — returns dict of image_id → {qvec, tvec}."""
    images = {}
    with open(path, 'rb') as f:
        num_images = struct.unpack('<Q', f.read(8))[0]
        for _ in range(num_images):
            img_id = struct.unpack('<i', f.read(4))[0]
            qvec = struct.unpack('<4d', f.read(32))
            tvec = struct.unpack('<3d', f.read(24))
            _ = struct.unpack('<i', f.read(4))[0]
            name = b''
            while True:
                c = f.read(1)
                if c == b'\x00': break
                name += c
            num_pts2d = struct.unpack('<Q', f.read(8))[0]
            f.read(24 * num_pts2d)
            images[img_id] = {'qvec': qvec, 'tvec': np.array(tvec)}
    return images

def qvec_to_R(qvec):
    qw, qx, qy, qz = qvec
    return np.array([
        [1 - 2*(qy**2 + qz**2), 2*(qx*qy - qw*qz), 2*(qx*qz + qw*qy)],
        [2*(qx*qy + qw*qz), 1 - 2*(qx**2 + qz**2), 2*(qy*qz - qw*qx)],
        [2*(qx*qz - qw*qy), 2*(qy*qz + qw*qx), 1 - 2*(qx**2 + qy**2)]
    ])

sparse_dir = 'data/sparse/0'
images = read_images_binary(f'{sparse_dir}/images.bin')
total_frames = len(glob.glob('data/images/*.jpg'))

print(f'Total input frames  : {total_frames:,}')
print(f'Registered cameras  : {len(images):,}  ({100*len(images)/total_frames:.1f}%)')

centers = np.array([
    -qvec_to_R(img['qvec']).T @ img['tvec']
    for img in images.values()
])

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(centers[:, 0], centers[:, 1], centers[:, 2], s=3, c='steelblue', alpha=0.7)
ax.set_title(f'Reconstructed camera trajectory — {len(images):,} registered cameras')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.tight_layout()
plt.show()

print(f'\nCamera center bounding box:')
print(f'  X: [{centers[:,0].min():.2f}, {centers[:,0].max():.2f}]')
print(f'  Y: [{centers[:,1].min():.2f}, {centers[:,1].max():.2f}]')
print(f'  Z: [{centers[:,2].min():.2f}, {centers[:,2].max():.2f}]')

### 📌 Discussion: Why Is `colmap image_undistorter` Commented Out? (LO3)

```python
# !colmap image_undistorter \
#     --image_path images \
#     --input_path sparse/0 \
#     --output_path ./ \
#     --output_type COLMAP
```

`colmap image_undistorter` rectifies all images to an ideal pinhole camera model and creates an `undistorted/`
directory needed for **dense reconstruction** pipelines such as:
- COLMAP’s own MVS (`colmap patch_match_stereo`)
- OpenMVS, MVSNet, or DepthMap

Dense reconstruction estimates a depth map for every pixel — it needs geometrically consistent images where
the epipolar lines are horizontal and all cameras share the same intrinsics.

**3DGS does not need this step.** The `train.py` script reads camera intrinsics directly from `cameras.bin`
and applies the radial distortion correction analytically during Gaussian projection. It never looks at
an undistorted image file — it reads the original `images/` directly.

Skipping undistortion saves ~5 minutes of compute and ~500 MB of disk space.

### 📌 Discussion: Why Is `colmap model_converter --output_type TXT` Commented Out? (LO2)

```python
# !colmap model_converter \
#   --input_path sparse/0 \
#   --output_path sparse/0 \
#   --output_type TXT
```

COLMAP writes its reconstruction in **binary format** (`.bin`) for performance — the files are compact and
fast to read with `struct.unpack`, as demonstrated in the §5 inspection cell above.

Converting to **text format** (`.txt`) produces human-readable ASCII files:
- `cameras.txt` — camera ID, model, width, height, focal length, principal point
- `images.txt` — image ID, quaternion, translation, camera ID, filename, 2D keypoint locations
- `points3D.txt` — 3D point ID, X/Y/Z, RGB color, reprojection error, track (image_id, keypoint_id)

This is primarily a **debugging tool** for Lab Objective 2: you can open `images.txt` in a text editor
to check whether specific cameras were registered, or inspect `points3D.txt` to see reprojection errors
on a per-point basis without writing any code.

In a production run we skip this because our Python inspection code reads the binary files directly.

### 📌 Discussion: Why Is `convert.py` Commented Out? (LO1)

```python
# !python ../gaussian-splatting/convert.py -s ./
```

`gaussian-splatting/convert.py` was written for the original Gaussian Splatting paper’s assumed workflow:

1. Run `colmap feature_extractor` + `colmap mapper` → `sparse/0/`
2. Run `colmap image_undistorter` → `images/`, `sparse/` (undistorted)
3. Run `convert.py` → renames directories, converts binary→text, creates `sparse/0/` in the expected layout
4. Train on the converted output

The modern `train.py` (the version we cloned) can read a COLMAP `sparse/0/` reconstruction **directly**
without any conversion step. It detects whether the input is a COLMAP reconstruction by looking for
`sparse/0/cameras.bin` (or `cameras.txt`) and reads intrinsics and extrinsics from there.

Since our GLOMAP output is already in `data/sparse/0/` in COLMAP binary format, `train.py -s data` finds
everything it needs without conversion. `convert.py` is a **historical artifact** of a transitional period
when the trainer assumed an older fixed directory structure.

**The streamlined pipeline**: GLOMAP writes COLMAP-format binary → `train.py` reads it directly. No intermediate
conversion, no undistortion, no `convert.py`.

## §6 — 3DGS Training

With the GLOMAP sparse reconstruction in `data/sparse/0/`, we can train 3DGS directly using `train.py -s data`.
The `-s data` flag tells the trainer to look for images in `data/images/` and the SfM model in `data/sparse/0/`.

**Training configuration (defaults):**
- Iterations: 30,000
- Densification interval: 100 (up to iteration 15,000)
- Opacity reset interval: 3,000
- Learning rate schedule: position 0.00016 → 0.0000016, SH 0.0025, opacity 0.05

**Reference results** (Bamburgh Castle, T4):
- Iteration 7,000: L1 = 0.01829, PSNR = 27.75 dB
- Iteration 30,000: L1 = 0.01436, PSNR = **29.63 dB**

> Note: PSNR is evaluated on the *training* set (no train/test split in this run). The `--train_test_exp` flag
> in the extended workflow below holds out test cameras for a more rigorous evaluation.

In [ ]:
with stage('3DGS training — 30,000 iterations', 'output/*/point_cloud/iteration_30000/point_cloud.ply'):
    !python gaussian-splatting/train.py -s data

## §7 — Pipeline Timing Summary

Compare your elapsed times against the reference run in the header. Differences are expected
depending on Colab GPU allocation and network speed.

In [ ]:
timing_summary()

## §8 — PLY Format Analysis (**Lab Part A**)

3DGS training outputs a `.ply` file with a **59-property schema** per Gaussian. This is very different from
a standard point-cloud PLY (3 position + 3 color properties). We need a **field-aware dispatcher** that
routes each PLY type to the correct loader.

**3DGS PLY schema** (59 fields × 4 bytes = 236 bytes/Gaussian in float32):
- `x, y, z` — position (3)
- `nx, ny, nz` — unused normals (3)
- `f_dc_0, f_dc_1, f_dc_2` — DC spherical harmonics (3)
- `f_rest_0 … f_rest_44` — higher-order SH coefficients, degree 1–3 (45)
- `opacity` — pre-sigmoid opacity logit (1)
- `scale_0, scale_1, scale_2` — log-space Gaussian scales (3)
- `rot_0, rot_1, rot_2, rot_3` — unit quaternion (4)

**Total**: 3 + 3 + 48 + 1 + 3 + 4 = **63 fields** (some implementations differ; inspect yours below).

In [ ]:
import glob
import numpy as np
from plyfile import PlyData

ply_paths = sorted(glob.glob('output/*/point_cloud/iteration_30000/point_cloud.ply'))
assert ply_paths, 'No PLY found — did training complete?'
ply_path = ply_paths[-1]
print(f'Loading: {ply_path}')

plydata = PlyData.read(ply_path)
vertex = plydata['vertex']
fields = [p.name for p in vertex.properties]
N = len(vertex)

print(f'\nGaussian count : {N:,}')
print(f'Fields ({len(fields)}): {fields}')
print(f'\nFile size (float32 PLY): {N * len(fields) * 4 / 1e6:.1f} MB')
print(f'  = {N:,} × {len(fields)} fields × 4 bytes')

In [ ]:
def detect_ply_type(plydata):
    """Detect whether a PLY is a 3DGS output or a simple point cloud.

    Returns: 'gaussian_splatting' | 'simple_pointcloud'
    """
    fields = {p.name for p in plydata['vertex'].properties}
    if 'opacity' in fields and 'f_dc_0' in fields:
        return 'gaussian_splatting'
    return 'simple_pointcloud'

ply_type = detect_ply_type(plydata)
print(f'Detected PLY type: {ply_type}')

import io
from plyfile import PlyElement

pts = np.zeros(5, dtype=[('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
                          ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')])
simple_ply = PlyData([PlyElement.describe(pts, 'vertex')])
print(f'Synthetic simple PLY type: {detect_ply_type(simple_ply)}')

## 9 — Sort Key Derivation (**Lab Part B**)

The `.splat` format stores Gaussians sorted by **opacity-weighted volume** (descending), so that a
renderer doing a single front-to-back pass can stop early once the accumulated alpha reaches 1.

The sort key is:

$$
k_i = \exp(s_0 + s_1 + s_2) \times \sigma(o_i)
$$

where

$$
s_j = \log(\text{scale}_j)
$$

are the log-space scales (so $\exp(s_j)$ is the activated scale), and

$$
\sigma(o_i) = \frac{1}{1 + e^{-o_i}}
$$

is the sigmoid-activated opacity.

**Equivalence**:

$$
\exp(s_0 + s_1 + s_2) = \exp(s_0)\cdot\exp(s_1)\cdot\exp(s_2)
$$

— the product of the three activated scales, i.e., the **volume** of the Gaussian ellipsoid (up to a $\frac{4}{3}\pi$ constant).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

s0 = np.array(vertex['scale_0'], dtype=np.float32)
s1 = np.array(vertex['scale_1'], dtype=np.float32)
s2 = np.array(vertex['scale_2'], dtype=np.float32)
opac = np.array(vertex['opacity'], dtype=np.float32)

volume = np.exp(s0 + s1 + s2)
sigmoid_opac = 1.0 / (1.0 + np.exp(-opac))
sort_key = volume * sigmoid_opac

sample = slice(0, 1000)
lhs = np.exp(s0[sample] + s1[sample] + s2[sample])
rhs = np.exp(s0[sample]) * np.exp(s1[sample]) * np.exp(s2[sample])
print(f'Sum-of-logs vs product-of-exps equivalence (max abs diff): {np.max(np.abs(lhs - rhs)):.2e}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(np.log10(sort_key + 1e-30), bins=100, color='steelblue', edgecolor='none')
axes[0].set_xlabel('log₁₀(sort key)'); axes[0].set_ylabel('Count')
axes[0].set_title('Sort key distribution (log scale)')
axes[1].hist(sigmoid_opac, bins=100, color='coral', edgecolor='none')
axes[1].set_xlabel('Sigmoid opacity'); axes[1].set_ylabel('Count')
axes[1].set_title('Opacity distribution')
plt.tight_layout()
plt.show()

print(f'\nSort key statistics:')
print(f'  min   : {sort_key.min():.4e}')
print(f'  median: {np.median(sort_key):.4e}')
print(f'  max   : {sort_key.max():.4e}')
print(f'  heavy-tailed? {(sort_key > 10 * np.median(sort_key)).sum():,} Gaussians are >10× the median')

## §10 — Compression Analysis (**Lab Part C**)

We measure file sizes at each compression stage:
1. Raw float32 `.ply` — N × (number of fields) × 4 bytes
2. `.splat` — N × 32 bytes (position 12B + scales 12B + rgba 4B + quaternion 4B)
3. `.splat` gzip-compressed — actual measured size

Compare to the SOG (Self-Organizing Gaussians) encoding from Module 5:
- SOG uses SOM spatial ordering to reduce entropy before gzip, achieving better compression than
  an opacity-volume sorted `.splat`.
- The byte-level entropy of a SOM-ordered file is lower because spatially adjacent Gaussians
  tend to have similar attributes — neighboring bytes are correlated → gzip finds longer matches.

In [ ]:
import os
import gzip
import struct
import numpy as np
from plyfile import PlyData

SH_C0 = 0.28209479177387814  # 1 / (2 * sqrt(pi))

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def ply_to_splat_bytes(plydata):
    """Convert 3DGS PLY to .splat binary (32 bytes/Gaussian, sorted by opacity-weighted volume)."""
    v = plydata['vertex']
    N = len(v)
    pos = np.column_stack([v['x'], v['y'], v['z']]).astype(np.float32)
    scales = np.exp(np.column_stack([v['scale_0'], v['scale_1'], v['scale_2']])).astype(np.float32)
    rgb = (0.5 + SH_C0 * np.column_stack([v['f_dc_0'], v['f_dc_1'], v['f_dc_2']])) * 255
    rgb = np.clip(rgb, 0, 255).astype(np.uint8)
    alpha = (sigmoid(np.array(v['opacity'])) * 255).astype(np.uint8)
    rgba = np.column_stack([rgb, alpha])
    q = np.column_stack([v['rot_0'], v['rot_1'], v['rot_2'], v['rot_3']]).astype(np.float32)
    q_norm = q / (np.linalg.norm(q, axis=1, keepdims=True) + 1e-9)
    q_bytes = ((q_norm * 128) + 128).astype(np.uint8)
    sk = np.exp(np.array(v['scale_0']) + np.array(v['scale_1']) + np.array(v['scale_2'])) \
         * sigmoid(np.array(v['opacity']))
    order = np.argsort(-sk)
    buf = bytearray(N * 32)
    ps = pos[order]; ss = scales[order]; rs = rgba[order]; qs = q_bytes[order]
    for i in range(N):
        o = i * 32
        struct.pack_into('<3f', buf, o,      *ps[i])
        struct.pack_into('<3f', buf, o + 12, *ss[i])
        buf[o + 24:o + 28] = rs[i].tobytes()
        buf[o + 28:o + 32] = qs[i].tobytes()
    return bytes(buf)

num_fields = len([p.name for p in plydata['vertex'].properties])
N = len(plydata['vertex'])
ply_size = os.path.getsize(ply_path)
splat_bytes = ply_to_splat_bytes(plydata)
splat_size = len(splat_bytes)
gzip_data = gzip.compress(splat_bytes, compresslevel=9)
gzip_size = len(gzip_data)

print(f'Gaussians              : {N:,}')
print(f'Fields per Gaussian    : {num_fields}')
print()
print(f'Float32 PLY (on disk)  : {ply_size / 1e6:.2f} MB')
print(f'Theoretical (N×fields×4): {N * num_fields * 4 / 1e6:.2f} MB  (actual may differ due to PLY header)')
print(f'.splat (32 B/Gaussian) : {splat_size / 1e6:.2f} MB')
print(f'.splat gzipped         : {gzip_size / 1e6:.2f} MB')
print()
print(f'Compression ratios:')
print(f'  PLY → .splat         : {ply_size / splat_size:.2f}×  ({num_fields * 4}/{32} bytes per Gaussian)')
print(f'  .splat → gzip(.splat): {splat_size / gzip_size:.2f}×')
print(f'  PLY → gzip(.splat)   : {ply_size / gzip_size:.2f}×')
print()
print('SOG comparison (Module 5):')
print('  SOM-ordered .splat gzip achieves ~2-3× better compression than opacity-volume sorted .splat gzip')
print('  because SOM ordering clusters spatially adjacent Gaussians, reducing byte entropy')
print('  and giving LZ77 longer backward-reference matches.')

In [ ]:
splat_path = ply_path.replace('.ply', '.splat')
with open(splat_path, 'wb') as f:
    f.write(splat_bytes)
print(f'Saved: {splat_path}')
print(f'Size : {os.path.getsize(splat_path) / 1e6:.2f} MB')

## §11 — WebGL Viewer & Artifact Diagnosis (**Lab Part D**)

Download the `.splat` file from your Colab session and drag it into:
- **antimatter15’s viewer**: https://antimatter15.com/splat/
- **Reference result**: https://nickmccarty.me/bamburgh-castle-splat

### Artifact Checklist

Inspect your rendering and check which artifacts are visible. For each, hypothesise the source:

| Artifact | Likely Source | How to Verify |
|---|---|---|
| Floating blobs in sky / water | Missing camera coverage (unregistered 519 frames) | Check camera plot in §5 — are sky-facing cameras absent? |
| Needle-like Gaussians on flat surfaces | Under-constrained 3D points (low parallax) | Check track length distribution — short tracks → weak triangulation |
| color fringing at object edges | Static sort approximation (opacity-weighted volume ≠ true depth order) | Rotate in viewer — does fringing rotate with the camera? |
| Blurry / over-smoothed surfaces | Insufficient training iterations (7k vs 30k) | Compare 7k and 30k PLY files |
| Dark patches | Opacity collapse during densification (Gaussians pruned below threshold) | Inspect `output/*/cfg_args` for `opacity_threshold` value |

### Download from Colab

```python
from google.colab import files
files.download(splat_path)
```

---
## §Extended — Depth Priors + Sparse Adam

The streamlined pipeline above achieves PSNR 29.63 dB in ~45 minutes with no depth supervision.
This section runs the **depth-prior workflow** documented in the supplemental How-To notebook,
which adds:

1. **Depth-Anything-V2 ViT-L** — monocular depth estimation for every frame (~35 min)
2. **`make_depth_scale.py`** — affine alignment of relative depth maps to metric scale (~2 min)
3. **Sparse Adam optimizer** — momentum-based, faster convergence on sparse Gaussians
4. **`--train_test_exp`** — holds out test cameras for rigorous PSNR evaluation

Reference result: PSNR 29.02 dB — slightly lower than the streamlined path by this metric,
but depth priors can improve quality on thin structures and weakly-textured surfaces that
GLOMAP triangulates poorly.

> **When to use depth priors:** If your GLOMAP registration rate is low (<40%) or your
> scene has large textureless regions (white walls, sky, water), depth priors provide
> useful regularisation. For well-textured scenes with good coverage, the streamlined
> path is faster and often matches or exceeds depth-prior quality.

In [ ]:
# ── Install Depth-Anything-V2 (only needed for extended workflow)
!git clone https://github.com/DepthAnything/Depth-Anything-V2.git
!wget https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth
!mkdir Depth-Anything-V2/checkpoints
!mv depth_anything_v2_vitl.pth Depth-Anything-V2/checkpoints/depth_anything_v2_vitl.pth

In [ ]:
with stage('Depth-Anything-V2 inference', 'data/depths/ — 1,031 relative depth maps'):
    !mkdir data/depths
    %cd Depth-Anything-V2
    !python run.py \
        --encoder vitl \
        --img-path ../data/images \
        --outdir ../data/depths \
        --pred-only \
        --grayscale
    %cd ..

In [ ]:
# make_depth_scale.py aligns the relative depth maps to metric scale using the
# triangulated 3D points from GLOMAP as ground truth — computing per-image
# scale and shift parameters (affine alignment in log-depth space).
with stage('Depth affine alignment', 'data/depth_scale.json — per-image scale/shift'):
    %cd gaussian-splatting/utils
    !python make_depth_scale.py \
        --base_dir ../../data \
        --depths_dir ../../data/depths
    %cd ../../

In [ ]:
with stage('3DGS training — depth priors + sparse_adam', 'output/*/point_cloud/iteration_30000/point_cloud.ply'):
    !python gaussian-splatting/train.py \
        -s data \
        --optimizer_type sparse_adam \
        --train_test_exp

---
## §15 — SOGS Comparison (Self-Organizing Gaussians)

The supplemental **SOGS** notebook (`Bamburgh_Castle_SOGS_Workflow.ipynb`) demonstrates an alternative
encoding pipeline from Module 5 (Permutation Learning & Encoding):

| Property | Standard `.splat` | SOGS |
|---|---|---|
| Gaussian order | Opacity-weighted volume sort | SOM spatial order |
| SH encoding | Full float32 | Quantised SH (lower bit-depth) |
| Gzip compression ratio | ~1.5–2× | ~3–5× (lower entropy from spatial order) |
| Attribute quantisation | None (float32 scales/quats) | Yes (8-bit quats, clustered scales) |
| Training framework | gaussian-splatting train.py | Custom SOGS trainer + Hydra config |

**Why SOGS compresses better**: After SOM ordering, adjacent Gaussians in the file are spatially
adjacent in the scene. Their attributes (position, color, scale) are similar, so the byte sequence
has lower entropy and LZ77 (used by gzip/zlib) finds longer backward-reference matches.

See Module 5 for the mathematical derivation: `GradSort` + `ShuffleSoftSort` for differentiable
permutation learning, and the SOM training procedure that produces the spatial order.

In [ ]:
import math

def byte_entropy(data: bytes) -> float:
    """Shannon entropy of byte distribution (bits per byte)."""
    counts = [0] * 256
    for b in data:
        counts[b] += 1
    n = len(data)
    return -sum((c/n) * math.log2(c/n) for c in counts if c > 0)

standard_entropy = byte_entropy(splat_bytes)
print(f'Standard .splat byte entropy : {standard_entropy:.3f} bits/byte')
print(f'  (max possible = 8.000 bits/byte for uniform distribution)')
print(f'  (lower entropy → better gzip compression)')
print()
print(f'For comparison, a SOGS-ordered .splat typically achieves ~6.5–7.0 bits/byte')
print(f'due to spatial correlation between adjacent Gaussians in the SOM ordering.')

---
## Lab 6 Completion Checklist

- [ ] **LO1** — Pipeline ran end-to-end; timing DataFrame matches reference order of magnitude
- [ ] **LO2** — Camera registration rate computed; trajectory plot shows coherent arc
- [ ] **LO3** — All four commented-out cells explained (mapper, undistorter, converter, convert.py)
- [ ] **LO4** — Training completed; PSNR at 7k and 30k iterations recorded
- [ ] **LO5** — Sort key equivalence verified; distribution plotted; heavy-tail confirmed
- [ ] **LO6** — `detect_ply_type` dispatcher implemented; compression ratios measured; .splat downloaded and viewed

**Deliverable**: Drop your `.splat` file into https://antimatter15.com/splat/ and document
one artifact you observe, with a hypothesis about which pipeline stage caused it.